In [ ]:
# Story Step 1: Mana journey start cheyyadaniki first tools bag ready chesukuntunnam.
! pip install crewai crewai-tools python-dotenv

In [ ]:
# Story Step 2: Ippudu model friends tho matlaadadaniki extra LiteLLM support install chestunnam.
! pip install "crewai[litellm]"

In [4]:
# Story Step 3: Key box open cheyyadaniki os ni pilustunnam.
import os
# Story Step 4: .env locker nundi secrets thisukodaniki dotenv helper ni pilustunnam.
from dotenv import load_dotenv

# Story Step 5: Locker open chesi keys ni memory lo pettukuntunnam.
load_dotenv()
# Story Step 6: GROQ key dorikinda leda ani chinna check chestunnam.
print(os.getenv("GROQ_API_KEY") is not None)

True


In [1]:
# Story Step 7: Malli os ni use chesi inko key ni choodadaniki prepare avtunnam.
import os
# Story Step 8: dotenv helper tho secret locker ni malli open cheyyadaniki ready.
from dotenv import load_dotenv

# Story Step 9: SERPER key kuda runtime ki load chesukuntunnam.
load_dotenv()

# Story Step 10: Search ki use ayye SERPER key value unda ani choostunnam.
print(os.getenv("SERPER_API_KEY"))

882c1f7575f2a47f97270364f74aae46ffac5eec


In [8]:
# Story Step 11: CrewAI hero ni scene lo ki teesukostunnam.
import crewai
# Story Step 12: Ee hero ye version lo undho choosi, mana notes tho match chestunnam.
print(crewai.__version__)

1.15.7


In [ ]:
# Story Step 13: Ippudu brain create cheyyadaniki LLM class ni import chestunnam.
from crewai import LLM
# Story Step 14: API key read cheyyadaniki os friend ni pilustunnam.
import os
# Story Step 15: Secret locker open cheyyadaniki dotenv helper ni ready chestunnam.
from dotenv import load_dotenv

# Story Step 16: .env nundi keys ni load chestunnam.
load_dotenv()

# Story Step 17: Mana first AI brain object ni create chestunnam.
llm = LLM(
    # Story Step 18: Brain e model meeda think cheyyalo ikkada cheptunnam.
    model="groq/llama-3.1-8b-instant",
    # Story Step 19: Brain door open avvadanki key pass chestunnam.
    api_key=os.getenv("GROQ_API_KEY")
# Story Step 20: Brain setup complete chesi block close chestunnam.
)

# Story Step 21: Chinna trial question adigi brain work avtunda test chestunnam.
response = llm.call("What is AI?")
# Story Step 22: Brain ichina answer ni screen meeda chupistunnam.
print(response)

In [ ]:
# Story Chapter 1: Ippudu strong version code run chestam, errors ni soft ga handle cheyyadaniki helpers import chestunnam.
import os
import re
import asyncio
import random
from dotenv import load_dotenv

# Story Chapter 2: Crew build cheyyadaniki main classes ni stage meeda ki teesukostunnam.
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

# Story Chapter 3: Groq + CrewAI cache glitch avoid cheyyadaniki patch apply chestunnam.
import crewai.llms.cache as crewai_cache
crewai_cache.mark_cache_breakpoint = lambda msg: msg

# Story Chapter 4: Secret locker (.env) open chesi keys ni memory lo load chestunnam.
load_dotenv()

# Story Chapter 5: Groq key mandatory kabatti mundu validate chestunnam.
groq_key = os.getenv("GROQ_API_KEY")
if not groq_key:
    raise ValueError("GROQ_API_KEY missing. .env lo correct key set cheyyi.")

# Story Chapter 6: Serper key optional; invalid/unauthorized aina app crash kakunda search ni off chestam.
serper_key = os.getenv("SERPER_API_KEY")
search_enabled = bool(serper_key and serper_key.strip())
if search_enabled:
    print("SERPER key detected. First search mode ON.")
else:
    print("SERPER_API_KEY missing/empty. Web search lekunda run chestunnam.")

# Story Chapter 7: Team andaru share chesukune common AI brain create chestunnam.
llm = LLM(
    model="groq/llama-3.1-8b-instant",
    api_key=groq_key,
    temperature=0.2
)

# Story Chapter 8: Ee function lo enable_search flag batti researcher ki tool attach chestam.
def build_blog_crew(topic: str, enable_search: bool = True):
    local_search_tool = SerperDevTool() if enable_search else None

    researcher = Agent(
        role="Senior Research Agent",
        goal="Research the topic and collect accurate, useful information.",
        backstory=(
            "You are an expert researcher. You collect key facts, trends, "
            "examples, and references from reliable sources."
        ),
        tools=[local_search_tool] if local_search_tool else [],
        llm=llm,
        verbose=True
    )

    analyst = Agent(
        role="Content Analyst Agent",
        goal="Analyze the research and extract the most important insights.",
        backstory=(
            "You are an expert content strategist. You convert raw research "
            "into structured insights, key points, and a clear content outline."
        ),
        llm=llm,
        verbose=True
    )

    writer = Agent(
        role="Blog Writer Agent",
        goal="Write a beginner-friendly blog post from the analyzed insights.",
        backstory=(
            "You are a clear and practical blog writer. You explain complex "
            "topics in simple language with examples."
        ),
        llm=llm,
        verbose=True
    )

    editor = Agent(
        role="Editor Agent",
        goal="Improve clarity, grammar, flow, and final presentation.",
        backstory=(
            "You are an experienced editor. You polish content and make it "
            "easy to read, structured, and professional."
        ),
        llm=llm,
        verbose=True
    )

    research_task = Task(
        description=(
            f"Research the topic: {topic}. "
            "Collect important facts, beginner-friendly examples, and references. "
            "If web tools are unavailable, use reliable general knowledge and state assumptions clearly."
        ),
        expected_output=(
            "A structured research summary with key facts, examples, and references or clearly stated assumptions."
        ),
        agent=researcher
    )

    analysis_task = Task(
        description=(
            f"Analyze the research for the topic: {topic}. "
            "Create a clean blog outline with key sections and main points."
        ),
        expected_output=(
            "A blog outline with introduction, main sections, examples, and conclusion."
        ),
        agent=analyst,
        context=[research_task]
    )

    writing_task = Task(
        description=(
            f"Write a complete beginner-friendly blog post on: {topic}. "
            "Use the outline and research. Keep it clear, short, and practical."
        ),
        expected_output=(
            "A complete blog post with title, intro, headings, examples, and conclusion."
        ),
        agent=writer,
        context=[research_task, analysis_task]
    )

    editing_task = Task(
        description=(
            "Review and improve the blog post. Fix grammar, improve flow, "
            "make it readable, and add a strong final summary."
        ),
        expected_output="Final polished blog post ready to publish.",
        agent=editor,
        context=[writing_task]
    )

    crew = Crew(
        agents=[researcher, analyst, writer, editor],
        tasks=[research_task, analysis_task, writing_task, editing_task],
        process=Process.sequential,
        verbose=True
    )
    return crew

# Story Chapter 9: Error text lo nundi wait time extract cheyyadam.
def _extract_wait_seconds(error_text: str, default_wait: int = 20) -> int:
    match = re.search(r"try again in\s*([0-9]+(?:\.[0-9]+)?)s", error_text, flags=re.IGNORECASE)
    if match:
        return int(float(match.group(1))) + 2
    return default_wait

# Story Chapter 10: Ee helper Unauthorized/403 errors ni detect chestundi.
def _is_serper_auth_error(error_text: str) -> bool:
    txt = error_text.lower()
    return ("serper" in txt or "google.serper.dev" in txt) and ("403" in txt or "unauthorized" in txt)

# Story Chapter 11: Crew execution ni retry wrapper lo run chestam with exponential backoff + jitter.
async def run_with_retry(topic: str, retries: int = 5):
    global search_enabled
    attempt = 1
    while attempt <= retries:
        crew = build_blog_crew(topic, enable_search=search_enabled)
        try:
            return await crew.kickoff_async()
        except Exception as e:
            msg = str(e)

            # Serper unauthorized vaste immediate ga search off chesi same attempt ni tool lekunda malli run chestam.
            if search_enabled and _is_serper_auth_error(msg):
                print("Serper 403/Unauthorized detected. Ippudu search OFF mode lo continue chestunnam.")
                search_enabled = False
                continue

            is_rate_limit = "rate_limit" in msg.lower() or "ratelimiterror" in msg.lower()
            if not is_rate_limit or attempt == retries:
                raise

            base_wait = _extract_wait_seconds(msg, default_wait=20)
            backoff = base_wait * (2 ** (attempt - 1))
            jitter = random.randint(1, 4)
            wait_for = min(backoff + jitter, 120)
            print(f"Rate limit hit. Attempt {attempt}/{retries}. Waiting {wait_for}s and retrying...")
            await asyncio.sleep(wait_for)
            attempt += 1

# Story Chapter 12: Notebook nundi direct run chestunte ee block execute avtundi.
if __name__ == "__main__":
    topic = input("Enter blog topic: ").strip()
    if not topic:
        topic = "AI for school kids"

    result = await run_with_retry(topic=topic, retries=5)

    print("\n\n================ FINAL BLOG ================\n")
    print(result)

SERPER key detected. First search mode ON.


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.7                                                                                        │
│  Latest version:  1.15.10                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e0095198-97cf-4015-8005-537cbb383f68                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: telugu. Collect important facts, beginner-friendly examples, and references. If web  │
│  tools are unavailable, use reliable general knowledge and state assumptions clearly.                           │
│  ID: 1947696b-6ad9-4f70-b409-853dd0eea4a3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Agent                                                                                   │
│                                                                                                                 │
│  Task: Research the topic: telugu. Collect important facts, beginner-friendly examples, and references. If web  │
│  tools are unavailable, use reliable general knowledge and state assumptions clearly.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for     │
│  model `llama-3.1-8b-instant` in organization `org_01kw64ryxhfwqbyv1wd6e1rjgp` service tier `on_demand` on      │
│  tokens per minute (TPM): Limit 6000, Used 2569, Requested 4409. Please try again in 9.78s. Need more tokens?   │
│  Upgrade to Dev Tier today at                                                                                   │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Research the topic: telugu. Collect important facts, beginner-friendly examples, and references. If web  │
│  tools are unavailable, use reliable general knowledge and state assumptions clearly.                           │
│  Agent: Senior Research Agent                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 483fa949-eea6-430a-ac96-2a5d2bbe95ef                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Rate limit hit. Attempt 3/5. Waiting 46s and retrying...


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Architecture (Detailed) - Simple Telugu English Mix

Ee project ni oka mini content factory laga think cheyyi. Parts ila untayi:

1. Environment Layer
- `.env` lo secrets untayi: `GROQ_API_KEY`, `SERPER_API_KEY`.
- `load_dotenv()` vatini runtime lo load chestundi.
- Enduku: keys code lo hardcode cheyakunda safe ga unchadaniki.

2. Model Layer (LLM Brain)
- `LLM(model='groq/llama-3.1-8b-instant', ...)` create chestam.
- Idi main thinking engine. Agents andariki same brain use avtundi.
- `temperature=0.3` ante response stable ga untundi, randomness takkuva.

3. Tool Layer
- `SerperDevTool()` web search tool.
- Research agent internet nundi facts collect cheyadaniki use chestadu.
- LLM only memory meeda depend kakunda fresh info tisukostundi.

4. Agent Layer (Team Members)
- `researcher`: data collect chestadu.
- `analyst`: data ni order lo petti outline chestadu.
- `writer`: readable blog ga rayadam.
- `editor`: grammar, flow, polish.

5. Task Layer (Work Tickets)
- `research_task` -> facts and references
- `analysis_task` -> outline
- `writing_task` -> complete draft
- `editing_task` -> final polished version

6. Orchestration Layer (Manager)
- `Crew(...)` manager laga panichestundi.
- `process=Process.sequential` kabatti order strict: research -> analysis -> writing -> editing.
- Context passing vallana next task ki previous output input laga velthundi.

7. Execution Layer
- User topic istadu.
- `build_blog_crew(topic)` team + tasks ready chestundi.
- `await crew.kickoff_async()` complete pipeline run chestundi.
- Final ga blog print avtundi.

### Why this architecture is good
- Clear responsibility split (each agent ki oka clear job).
- Reusable structure (topic marchina same pipeline use cheyochu).
- Better quality through multi-step refinement.
- Easy debugging (ye stage lo issue vachindo identify cheyochu).

## Architecture Diagram + Flow Diagram

Below is a visual diagram of how your CrewAI project works end-to-end.

```mermaid
flowchart TD
    U[User Topic Input] --> E[Environment Layer\nload_dotenv + API keys]
    E --> M[Model Layer\nGroq LLM]
    E --> T[Tool Layer\nSerperDevTool]
    M --> A1[Researcher Agent]
    T --> A1
    M --> A2[Analyst Agent]
    M --> A3[Writer Agent]
    M --> A4[Editor Agent]

    A1 --> R1[Research Task\ncollect facts + references]
    R1 --> A2
    A2 --> R2[Analysis Task\ncreate outline]
    R2 --> A3
    A3 --> R3[Writing Task\ncreate full blog draft]
    R3 --> A4
    A4 --> R4[Editing Task\npolish final blog]
    R4 --> O[Final Blog Output]

    C[Crew Orchestrator\nProcess.sequential] -. manages order .-> R1
    C -. manages order .-> R2
    C -. manages order .-> R3
    C -. manages order .-> R4
```

### Super Simple Flow
1. Topic ivvadam
2. Research collect cheyadam
3. Outline ready cheyadam
4. Blog draft rayadam
5. Edit chesi polish cheyadam
6. Final blog output ivvadam

Flow line:
`Topic -> Research -> Analysis -> Writing -> Editing -> Final Blog`

## Flow Explanation - Like Explaining to a Kid

Imagine nuvvu school project chesthunnaav: "AI gurinchi blog raayali".
Nuvvu okkadive kaadhu, nee daggara 4 friends unnaru.

- Friend 1 (Researcher): "Nenu books/internet chusi facts collect chestha."
- Friend 2 (Analyst): "Nenu facts ni neat ga points ga arrange chestha."
- Friend 3 (Writer): "Nenu story laga easy ga article raastha."
- Friend 4 (Editor): "Nenu spelling, grammar, neat finish chestha."

Ippudu step-by-step chuddam:

1. Start
- Nuvvu topic chepthaav. Example: "What is AI for beginners".

2. Data Hunt
- Researcher web lo search chesi manchi info collect chestadu.

3. Sorting
- Analyst aa info ni sections ga break chestadu.
- Example sections: Intro, Real-life examples, Benefits, Conclusion.

4. Writing
- Writer aa sections base chesukoni full blog draft rayadam.

5. Polishing
- Editor mistakes remove chesi final version super readable ga chestadu.

6. Final Output
- Crew final blog ni one clean answer ga ichestundi.

### Very Simple Mental Model
- Topic = raw material
- Agents = workers
- Tasks = work steps
- Crew = manager
- Final blog = finished product

### End-to-end data flow
`User Topic -> Research Task -> Analysis Task -> Writing Task -> Editing Task -> Final Blog`

### Small practical tips
- Topic specific ga ivvu (example: "AI in healthcare for beginners").
- API keys correct ga set cheyyi lekapothe run avvadu.
- Same pipeline lo different topics test cheyyi quality compare cheyadaniki.

In [ ]:
# Story Note: Ee cell ni future mini experiments kosam empty ga reserve chesam.

In [ ]:
# Story Note: Ee last cell lo tarvata quick tests or ideas try cheyyachu.